# 🧑‍💼 Human-in-the-Loop với Deep Agents SDK

Notebook này demo **Human-in-the-Loop (HIL)** sử dụng LangChain **Deep Agents SDK** (`create_deep_agent`).

Thay vì build StateGraph thủ công (như notebook `hil-deepseek-agent.ipynb`), chúng ta dùng `create_deep_agent` — một high-level harness đã tích hợp sẵn:
- Task planning (`write_todos`)
- Virtual filesystem
- Subagent delegation
- **Human-in-the-loop** qua tham số `interrupt_on`

## Mục lục

1. Cài đặt dependencies
2. Định nghĩa tools (safe + dangerous)
3. Tạo Deep Agent với `interrupt_on`
4. Pattern cơ bản: Invoke → Interrupt → Resume
5. Console HIL loop: Approve / Reject / Edit
6. Batch multiple tool calls
7. Conditional interrupts
8. Best practices

## 1. Cài Đặt Dependencies

Cài `deepagents` package và các thư viện cần thiết.

In [28]:
import subprocess
import sys

deps = [
    "deepagents>=0.6.0",
    "langchain-openai>=0.3.0",
    "langgraph>=0.4.0",
    "langchain-core>=0.3.0",
]

for dep in deps:
    print(f"Installing {dep}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", dep])
print("✅ All dependencies installed!")

Installing deepagents>=0.6.0...
Installing langchain-openai>=0.3.0...
Installing langgraph>=0.4.0...
Installing langchain-core>=0.3.0...
✅ All dependencies installed!


## 2. Import & Cấu Hình Model

Import thư viện, định nghĩa tools, và khởi tạo model.

**Chọn model provider:**
- OpenAI: `openai:gpt-5.5` (cần `OPENAI_API_KEY`)
- DeepSeek (OpenAI-compatible): dùng `init_chat_model` với custom base_url
- Google: `google_genai:gemini-3.5-flash` (cần `GOOGLE_API_KEY`)

> Mặc định notebook này dùng OpenAI format. Bạn có thể đổi `model` string tuỳ ý.

In [29]:
import os
import json
from datetime import datetime
from getpass import getpass

from langchain_core.tools import tool
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
from deepagents import create_deep_agent


In [54]:
from langchain_deepseek import ChatDeepSeek
import dotenv
dotenv.load_dotenv()  # Load environment variables from .env file

# Initialize the DeepSeek V4 Pro flagship model
deepseek = ChatDeepSeek(
    model="deepseek-v4-flash",
    temperature=0.7,
    max_tokens=10000
)
deepseek.invoke('hi')

AIMessage(content='你好！有什么可以帮你的吗？😊', additional_kwargs={'refusal': None, 'reasoning_content': '好的，用户只发了一个“hi”，这是一个非常简单的问候。没有复杂的指令或问题。深层需求可能就是打个招呼，开启对话。我需要友好地回应，表示收到问候，并开放地邀请用户提出具体需求。可以用热情的问候回复，然后自然地询问需要什么帮助，为后续对话铺路。想到了用“你好！有什么可以帮你的吗？”这样的句式，既礼貌又直接。'}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 5, 'total_tokens': 100, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 84, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '818d69a5-32be-4159-9084-2008096b8c72', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f1227-823c-7580-a0a2-b1cc07172d7b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 95

In [ ]:
from langchain_openai import ChatOpenAI
import dotenv
dotenv.load_dotenv()  # Load environment variables from .env file

v_glm46 = ChatOpenAI(
    model="v_glm46",
    base_url="https://genai.vnpay.vn/aigateway/llm_glm46/v1",
    temperature=0.7,
    max_tokens=100000,
)

# Test thử
v_glm46.invoke("Hello")

AIMessage(content='\n\nHello! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 675, 'prompt_tokens': 11, 'total_tokens': 686, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'v_glm46', 'system_fingerprint': None, 'id': 'chatcmpl-b149188b51141c39', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f1223-a1c5-76e1-b399-be7d0493929e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 675, 'total_tokens': 686, 'input_token_details': {}, 'output_token_details': {}})

In [45]:
from langchain_openai import ChatOpenAI
import dotenv
dotenv.load_dotenv()  # Load environment variables from .env file

v_air45 = ChatOpenAI(
    model="v_air45",
    base_url="https://genai.vnpay.vn/aigateway/llm_glm_air/v1",
    temperature=0.7,
    max_tokens=100000,
)

# Test thử
v_air45.invoke("Hello")

AIMessage(content="Hello! 👋 How can I help you today? Feel free to ask me anything, whether it's a question, creative request, or just a conversation starter. I'm here to assist! 😊", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 245, 'prompt_tokens': 6, 'total_tokens': 251, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'v_air45', 'system_fingerprint': None, 'id': 'chatcmpl-8aaf69ed6485874e', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f1225-dcc3-71e0-94f4-f37180176263-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 245, 'total_tokens': 251, 'input_token_details': {}, 'output_token_details': {}})

In [55]:

# # ── Cấu hình API key ──
# # Ưu tiên: OPENAI_API_KEY > DEEPSEEK_API_KEY
# api_key = os.environ.get("OPENAI_API_KEY") or os.environ.get("DEEPSEEK_API_KEY")
# if not api_key:
#     api_key = getpass("🔑 Nhập API key của bạn: ")

MODEL_STRING = deepseek

# print(f"✅ Model: {MODEL_STRING}")
# print(f"✅ API Key: ...{api_key[-4:]}")

## 3. Định Nghĩa Tools

Định nghĩa 3 tools với mức độ "nguy hiểm" khác nhau:
- `search_database` — **Safe**, chạy tự do
- `send_email` — **Dangerous**, cần human approve/check/edit
- `delete_user` — **Very dangerous**, cần approval chặt chẽ

In [56]:
# ── Mock database ──
CUSTOMERS = {
    "C001": {"name": "Nguyen Van A", "email": "a@example.com", "plan": "Premium", "status": "active"},
    "C002": {"name": "Tran Thi B", "email": "b@example.com", "plan": "Basic", "status": "inactive"},
    "C003": {"name": "Le Van C", "email": "c@example.com", "plan": "Enterprise", "status": "active"},
}


@tool
def search_database(customer_id: str) -> str:
    """Search customer information by customer ID. Safe tool — no approval needed."""
    result = CUSTOMERS.get(customer_id.upper())
    if result:
        return json.dumps(result, indent=2, ensure_ascii=False)
    return json.dumps({"error": f"Customer {customer_id} not found"})


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient. REQUIRES HUMAN APPROVAL — check content before sending."""
    print(f"  📧 Sending email to {to}: {subject}")
    # In production: call SMTP API
    return f"✅ Email sent to {to}: '{subject}'"


@tool
def delete_user(customer_id: str, reason: str) -> str:
    """Delete a customer account. VERY DANGEROUS — requires strict human approval."""
    print(f"  🗑️ Deleting user {customer_id}: {reason}")
    return f"✅ User {customer_id} deleted. Reason: {reason}"


tools = [search_database, send_email, delete_user]
print(f"✅ {len(tools)} tools defined: {[t.name for t in tools]}")
print("   - search_database (safe)")
print("   - send_email (dangerous)")
print("   - delete_user (very dangerous)")

✅ 3 tools defined: ['search_database', 'send_email', 'delete_user']
   - search_database (safe)
   - send_email (dangerous)
   - delete_user (very dangerous)


## 4. Tạo Deep Agent với HIL

Sự khác biệt lớn nhất so với raw LangGraph: **chỉ một dòng `create_deep_agent`** thay vì phải build StateGraph thủ công.

Tham số `interrupt_on` là dictionary mapping **tên tool** → cấu hình interrupt:
- `True` — bật interrupt với đầy đủ decisions (approve, edit, reject, respond)
- `False` — tắt interrupt
- `dict` — custom config, chỉ định `allowed_decisions`

> ⚠️ **Bắt buộc:** phải có `checkpointer` (MemorySaver) để lưu state giữa interrupt & resume.

In [48]:
# Checkpointer là bắt buộc cho HIL
checkpointer = MemorySaver()

# Tạo Deep Agent với interrupt_on
agent = create_deep_agent(
    model=MODEL_STRING,
    tools=tools,
    checkpointer=checkpointer,
    interrupt_on={
        # Safe tool — không cần interrupt
        "search_database": False,
        # Dangerous — cho phép approve, edit, reject
        "send_email": {
            "allowed_decisions": ["approve", "edit", "reject"]
        },
        # Very dangerous — chỉ approve hoặc reject (không cho edit args)
        "delete_user": {
            "allowed_decisions": ["approve", "reject"]
        },
    },
    system_prompt="""You are a customer support agent.
You have access to customer database, email system, and account management.
For sensitive operations (email, deletion), wait for human approval.
Always explain what you're about to do before calling a tool.""",
)

# print("✅ Deep Agent created with HIL configuration!")
# print()
# print("Interrupt config:")
# for tool_name, config in agent.interrupt_on_config.items():
#     print(f"  • {tool_name}: {config}")

## 5. Pattern Cơ Bản — Invoke → Interrupt → Resume

Đây là luồng cốt lõi của HIL với Deep Agents SDK:

```
User input → Agent nhận task → Agent gọi tool (dangerous)
                                     ↓
                              ⛔ INTERRUPT — chờ human decision
                                     ↓
        ┌──────────────────────┬─────┴──────┬──────────────────┐
        │ ✅ Approve           │ ✏️ Edit     │ ❌ Reject        │
        │ (chạy tool gốc)     │ (sửa args)  │ (bỏ qua tool)    │
        └──────────────────────┴────────────┴──────────────────┘
                                     ↓
                              Agent tiếp tục → Final response
```

Cell dưới đây chạy agent với một yêu cầu đơn giản và xử lý interrupt.

In [62]:
def get_human_decision(action, review_cfg):
    """Hỏi user approve/edit/reject. KHÔNG có default — bắt buộc phải chọn."""
    name = action["name"]
    args = action["args"]
    allowed = review_cfg["allowed_decisions"]
    
    # Tự động approve search
    if name == "search_database":
        return {"type": "approve"}
    
    while True:
        if name == "send_email":
            print(f"\n  📧 Email preview:")
            print(f"     To: {args.get('to', '?')}")
            print(f"     Subject: {args.get('subject', '?')}")
            print(f"     Body: {args.get('body', '?')[:100]}")
            
            print(f"  Options: {', '.join(allowed)}")
            choice = input("  Your decision: ").strip().lower()
            
            if choice == "approve" or choice == "a":
                return {"type": "approve"}
            elif choice == "edit" or choice == "e":
                if "edit" not in allowed:
                    print("  ❌ Edit not allowed for this tool. Choose another option.")
                    continue
                new_to = input("  New recipient: ").strip() or args.get("to", "")
                new_subject = input("  New subject: ").strip() or args.get("subject", "")
                return {
                    "type": "edit",
                    "edited_action": {
                        "name": name,
                        "args": {"to": new_to, "subject": new_subject, "body": args.get("body", "")},
                    },
                }
            elif choice == "reject" or choice == "r":
                if "reject" not in allowed:
                    print("  ❌ Reject not allowed for this tool. Choose another option.")
                    continue
                reason = input("  Rejection reason (optional): ").strip()
                if not reason:
                    reason = f"User rejected {name}. Do not retry."
                return {"type": "reject", "message": reason}
            else:
                print(f"  ❌ Invalid choice '{choice}'. Must be one of: approve/a, edit/e, reject/r")
                continue
        
        if name == "delete_user":
            print(f"\n  ⚠️  DELETE USER: {args.get('customer_id', '?')}")
            print(f"      Reason: {args.get('reason', '?')}")
            
            print(f"  Options: {', '.join(allowed)}")
            choice = input("  Your decision: ").strip().lower()
            
            if choice == "approve" or choice == "a":
                if "approve" not in allowed:
                    print("  ❌ Approve not allowed for this tool. Choose another option.")
                    continue
                return {"type": "approve"}
            elif choice == "reject" or choice == "r":
                if "reject" not in allowed:
                    print("  ❌ Reject not allowed for this tool. Choose another option.")
                    continue
                return {
                    "type": "reject",
                    "message": f"User rejected deletion of {args.get('customer_id', 'unknown')}. Do not retry.",
                }
            else:
                print(f"  ❌ Invalid choice '{choice}'. Must be one of: approve/a, reject/r")
                continue
        
        # Fallback for unknown tools
        print(f"\n  🔧 Unknown tool '{name}'. Approving with original args.")
        return {"type": "approve"}


def run_agent_with_hil(user_input: str, thread_id: str):
    """Run agent and handle HIL interrupt — approve, reject, or edit tool calls."""
    config = {"configurable": {"thread_id": thread_id}}
    
    print(f"👤 User: {user_input}")
    print("─" * 60)
    
    # ── Bước 1: Invoke agent ──
    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config,
        version="v2",
    )
    
    # ── Bước 2: Kiểm tra interrupt ──
    if not result.interrupts:
        print("✅ Agent hoàn thành ngay (không cần human review)")
        print(f"\n🤖 Agent: {result.value['messages'][-1].content}")
        return
    
    # ── Bước 3: Xử lý interrupt ──
    interrupt_value = result.interrupts[0].value
    action_requests = interrupt_value["action_requests"]
    review_configs = {cfg["action_name"]: cfg for cfg in interrupt_value["review_configs"]}
    
    print(f"\n⛔ Agent đã dừng — cần duyệt {len(action_requests)} tool call(s):")
    print("─" * 60)
    
    decisions = []
    for i, action in enumerate(action_requests, 1):
        cfg = review_configs[action["name"]]
        print(f"\n  #{i} 🔧 {action['name']}")
        print(f"     Args: {json.dumps(action['args'], indent=6)}")
        print(f"     Allowed: {cfg['allowed_decisions']}")
        
        decision = get_human_decision(action, cfg)
        decisions.append(decision)
    
    print("\n" + "─" * 60)
    print(f"📋 Decisions: {json.dumps(decisions, indent=2, ensure_ascii=False)}")
    
    # ── Bước 4: Resume agent ──
    result = agent.invoke(
        Command(resume={"decisions": decisions}),
        config=config,
        version="v2",
    )
    
    print("\n" + "─" * 60)
    print(f"🤖 Agent: {result.value['messages'][-1].content}")


print("✅ Helper functions defined (fixed HIL logic)!")
print("Run the next cell to test.")

✅ Helper functions defined (fixed HIL logic)!
Run the next cell to test.


## 6. Chạy Thử — Demo Tương Tác

Chạy cell này để test HIL flow. Agent sẽ:

1. Search database (safe → chạy tự do)
2. Gửi email (dangerous → chờ bạn duyệt)
3. Xoá user (very dangerous → chờ bạn duyệt)

Bạn sẽ thấy console prompt hỏi **approve/edit/reject** từng tool.

In [66]:
# ── Test 1: Yêu cầu hỗn hợp (safe + dangerous) ──
run_agent_with_hil(
    user_input="Look up customer C003, then send them a welcome email with subject 'Welcome to Premium' "
                "and body 'Thank you for choosing us!'",
    thread_id="demo4",
)

👤 User: Look up customer C003, then send them a welcome email with subject 'Welcome to Premium' and body 'Thank you for choosing us!'
────────────────────────────────────────────────────────────

⛔ Agent đã dừng — cần duyệt 1 tool call(s):
────────────────────────────────────────────────────────────

  #1 🔧 send_email
     Args: {
      "to": "c@example.com",
      "subject": "Welcome to Premium",
      "body": "Thank you for choosing us!"
}
     Allowed: ['approve', 'edit', 'reject']

  📧 Email preview:
     To: c@example.com
     Subject: Welcome to Premium
     Body: Thank you for choosing us!
  Options: approve, edit, reject

────────────────────────────────────────────────────────────
📋 Decisions: [
  {
    "type": "approve"
  }
]
  📧 Sending email to c@example.com: Welcome to Premium

────────────────────────────────────────────────────────────
🤖 Agent: 
I've successfully looked up customer C003 and sent them a welcome email.

Customer details:
- Name: Le Van C
- Email: c@example

## 7. Batch Multiple Tool Calls

Khi agent gọi **nhiều tool dangerous** cùng lúc, tất cả interrupt được gom vào một batch.
Bạn phải cung cấp decisions theo đúng thứ tự `action_requests`.

> 🔑 **Quan trọng:** Số lượng decisions **phải** bằng số lượng action_requests.

In [68]:
# ── Test 2: Batch multiple tool calls ──
# Agent sẽ gọi send_email + delete_user cùng lúc → 1 interrupt với 2 action_requests

def run_batch_demo():
    config = {"configurable": {"thread_id": "demo-hil-batch"}}
    user_input = "Send email to a@example.com subject 'Goodbye' body 'Sorry to see you go', then delete customer C002 because they requested account removal."
    
    print(f"👤 User: {user_input}")
    print("─" * 60)
    
    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config,
        version="v2",
    )
    
    if not result.interrupts:
        print("✅ No interrupts")
        return
    
    interrupt_value = result.interrupts[0].value
    action_requests = interrupt_value["action_requests"]
    
    print(f"\n⛔ Interrupt with {len(action_requests)} pending action(s):")
    for i, a in enumerate(action_requests, 1):
        print(f"  #{i}: {a['name']}({json.dumps(a['args'])})")
    
    # Decisions MUST match order of action_requests
    decisions = [
        # Approve email
        {"type": "approve"},
        # Reject deletion
        {
            "type": "reject",
            "message": "User rejected account deletion. Customer C002 should be kept inactive, not deleted.",
        },
    ]
    
    print(f"\n📋 Decisions: {json.dumps(decisions, indent=2, ensure_ascii=False)}")
    
    result = agent.invoke(
        Command(resume={"decisions": decisions}),
        config=config,
        version="v2",
    )
    
    print("\n" + "─" * 60)
    print(f"🤖 Agent: {result.value['messages'][-1].content}")


run_batch_demo()

👤 User: Send email to a@example.com subject 'Goodbye' body 'Sorry to see you go', then delete customer C002 because they requested account removal.
────────────────────────────────────────────────────────────

⛔ Interrupt with 1 pending action(s):
  #1: send_email({"to": "a@example.com", "subject": "Goodbye", "body": "Sorry to see you go"})

📋 Decisions: [
  {
    "type": "approve"
  },
  {
    "type": "reject",
    "message": "User rejected account deletion. Customer C002 should be kept inactive, not deleted."
  }
]


ValueError: Number of human decisions (2) does not match number of hanging tool calls (1).

## 8. Conditional Interrupt

Không phải lúc nào cũng cần interrupt. Dùng `when` predicate để chỉ interrupt khi **điều kiện cụ thể** xảy ra.

Ví dụ: chỉ interrupt `send_email` khi email gửi ra **ngoài domain nội bộ** (@example.com).

In [ ]:
from deepagents import create_deep_agent
from langchain.agents.middleware import ToolCallRequest


def email_to_external_domain(request: ToolCallRequest) -> bool:
    """Only interrupt if email is sent to an external domain (not @example.com)."""
    to_email = request.tool_call["args"].get("to", "")
    return not to_email.endswith("@example.com")


agent_conditional = create_deep_agent(
    model=MODEL_STRING,
    tools=tools,
    checkpointer=MemorySaver(),
    interrupt_on={
        "send_email": {
            "allowed_decisions": ["approve", "edit", "reject"],
            "when": email_to_external_domain,  # ← Conditional predicate
        },
        "delete_user": {"allowed_decisions": ["approve", "reject"]},
        "search_database": False,
    },
    system_prompt="You are a customer support agent.",
)

print("✅ Conditional interrupt agent created!")
print("   send_email chỉ interrupt khi email gửi ra ngoài @example.com")


# ── Test: Email nội bộ (không interrupt) ──
def test_conditional():
    config = {"configurable": {"thread_id": "demo-conditional"}}
    
    print("👉 Test: Send email to INTERNAL (a@example.com)")
    print("   → Dự đoán: KHÔNG interrupt (vì domain là @example.com)")
    print()
    
    result = agent_conditional.invoke(
        {"messages": [{"role": "user", "content": "Send email to a@example.com with subject 'Hi' and body 'Test'"}]},
        config=config,
        version="v2",
    )
    
    if result.interrupts:
        print("⛔ Interrupted!")
    else:
        print("✅ No interrupt — email nội bộ được gửi tự động!")
        print(f"\n🤖 Agent: {result.value['messages'][-1].content[:200]}")


test_conditional()

## 9. Best Practices & Tổng Kết

### So sánh: Raw LangGraph vs Deep Agents SDK

| Khía cạnh | Raw LangGraph | Deep Agents SDK |
|-----------|--------------|-----------------|
| **Code** | 50+ dòng (StateGraph, nodes, edges, compile) | 1 dòng `create_deep_agent()` |
| **HIL config** | `interrupt_before=["tools"]` thủ công | `interrupt_on={}` declarative |
| **Decisions** | Tự xử lý state management | `Command(resume={"decisions": [...]})` |
| **Conditional** | Viết custom node logic | `when` predicate |
| **Built-in** | Không có | Task planning, filesystem, subagents, memory |

### Best Practices

1. **Luôn dùng checkpointer** — `MemorySaver()` hoặc persistent store
2. **Dùng cùng thread_id** — Khi resume, config phải giống hệt lúc invoke
3. **Match decision order** — Decisions array phải cùng thứ tự với `action_requests`
4. **Reject message rõ ràng** — Nói cho agent biết "không chạy tool này, hãy làm gì tiếp theo"
5. **Edit một cách thận trọng** — Sửa args quá nhiều có thể khiến agent confused
6. **Phân loại rủi ro** — `approve+edit+reject` cho high-risk, `approve+reject` cho medium, `False` cho low-risk
7. **Conditional predicates** — Giảm noise, chỉ interrupt khi thực sự cần

### Khi nào dùng interrupt?

| Tình huống | Nên interrupt? |
|-----------|:-------------:|
| Search/Read database | ❌ Không |
| Gửi email notification | ✅ Có |
| Xoá dữ liệu | ✅ Có (strict) |
| Thanh toán / Refund | ✅ Có (strict) |
| Gọi API nội bộ | ⚠️ Tuỳ (có thể auto nếu idempotent) |
| Code execution | ✅ Có |